<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Low_DTE/LowDTEVectorEngine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

class LowDTEVectorEngine:
    def __init__(self, initial_capital=1000000):
        self.initial_capital = initial_capital

    def bsm_price(self, S, K, T, r, sigma, option_type='call'):
        """Vectorized Black-Scholes pricing function."""
        # Prevent division by zero or negative time
        T = np.maximum(T, 1e-5)
        sigma = np.maximum(sigma, 1e-4)

        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)

        if option_type == 'call':
            price = S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.pdf(d2)
            delta = stats.norm.cdf(d1)
        else:
            price = K * np.exp(-r * T) * stats.norm.cdf(-d2) - S * stats.norm.cdf(-d1)
            delta = stats.norm.cdf(d1) - 1

        return price, delta

    def generate_synthetic_data(self, num_days=1000):
        """Generates realistic synthetic daily paths with volatility regimes."""
        np.random.seed(42)

        # Simulate dynamic VIX environment (Mean-reverting random walk)
        vix = np.zeros(num_days)
        vix[0] = 18.0
        for i in range(1, num_days):
            vix[i] = max(9.0, vix[i-1] + 0.1 * (16.0 - vix[i-1]) + np.random.normal(0, 1.2))

        # Generate underlying asset returns scaled by VIX
        daily_vol = (vix / 100) / np.sqrt(252)
        returns = np.random.normal(0.0002, daily_vol)

        spy_prices = 500.0 * np.exp(np.cumsum(returns))

        # Simulate intraday maximum adverse excursion (MAE) to check stop-losses
        # Max high/low deviation within the day
        intraday_noise = np.random.exponential(scale=0.008, size=num_days)
        spy_high = spy_prices * (1 + intraday_noise * 0.6)
        spy_low = spy_prices * (1 - intraday_noise * 0.6)

        df = pd.DataFrame({
            'SPY_Open': spy_prices * (1 - np.random.normal(0, 0.002, num_days)),
            'SPY_High': spy_high,
            'SPY_Low': spy_low,
            'SPY_Close': spy_prices,
            'VIX': vix
        })
        return df

    def run_short_credit_spread_backtest(self, df, target_delta=0.20, spread_width=5.0, stop_loss_mult=3.0, vix_filter=25.0):
        """
        Executes a vectorized backtest of a short credit spread strategy.
        Assumes daily 0DTE allocation based on open conditions.
        """
        df = df.copy()
        r = 0.04  # Risk-free rate
        T_entry = 1.0 / 252.0  # 0DTE duration (1 trading day)

        # 1. Regimes / Volatility Filters
        # Avoid entries if VIX crosses our high-gamma threshold
        df['Signal'] = np.where(df['VIX'] < vix_filter, 1, 0)

        # 2. Strike Selection Engine (Using ATM approximations via delta)
        S_open = df['SPY_Open'].values
        sigma_entry = (df['VIX'].values / 100)

        # Approximate short put strike using target delta
        # K = S * exp(-Z_delta * sigma * sqrt(T))
        z_score = stats.norm.ppf(target_delta)
        df['Short_Put_Strike'] = np.round(S_open * np.exp(z_score * sigma_entry * np.sqrt(T_entry)), 0)
        df['Long_Put_Strike'] = df['Short_Put_Strike'] - spread_width

        # 3. Premium Logic Engine (Entry Credit Calculation)
        short_premium, _ = self.bsm_price(S_open, df['Short_Put_Strike'].values, T_entry, r, sigma_entry, 'put')
        long_premium, _ = self.bsm_price(S_open, df['Long_Put_Strike'].values, T_entry, r, sigma_entry, 'put')

        df['Credit_Received'] = short_premium - long_premium
        # Safeguard against pricing errors or zero premiums
        df['Credit_Received'] = np.maximum(df['Credit_Received'], 0.05)

        # 4. Intraday Risk Wrapper (Stop Loss & Expiration Profiling)
        # Calculate worst-case premium spike using the day's Low price
        S_low = df['SPY_Low'].values
        T_expiry = 1e-5 # Close to zero at expiration/end of day

        short_p_at_low, _ = self.bsm_price(S_low, df['Short_Put_Strike'].values, T_expiry, r, sigma_entry, 'put')
        long_p_at_low, _ = self.bsm_price(S_low, df['Long_Put_Strike'].values, T_expiry, r, sigma_entry, 'put')
        max_intraday_spread_value = short_p_at_low - long_p_at_low

        # Determine if premium stop-loss was triggered mid-day
        stop_threshold = df['Credit_Received'].values * stop_loss_mult
        stop_triggered = max_intraday_spread_value > stop_threshold

        # Expiration values
        S_close = df['SPY_Close'].values
        short_p_final = np.maximum(df['Short_Put_Strike'].values - S_close, 0)
        long_p_final = np.maximum(df['Long_Put_Strike'].values - S_close, 0)
        final_spread_value = short_p_final - long_p_final

        # Compute Gross PnL per trade
        pnl_if_held = df['Credit_Received'].values - final_spread_value
        pnl_if_stopped = -df['Credit_Received'].values * (stop_loss_mult - 1)

        # Apply maximum defined risk cap (Spread Width minus credit received)
        max_loss_cap = -(spread_width - df['Credit_Received'].values)
        pnl_if_stopped = np.maximum(pnl_if_stopped, max_loss_cap)

        # Assemble raw returns vector
        trade_pnl = np.where(stop_triggered, pnl_if_stopped, pnl_if_held)
        df['Trade_PnL'] = trade_pnl * df['Signal'] # Apply volatility filter

        return self.generate_metrics_report(df)

    def generate_metrics_report(self, df):
        """Computes advanced quantitative metrics from the PnL vector."""
        pnl = df['Trade_PnL'].values
        active_trades = pnl[df['Signal'] == 1]

        wins = active_trades > 0
        win_rate = np.sum(wins) / len(active_trades) if len(active_trades) > 0 else 0

        gross_profits = np.sum(active_trades[active_trades > 0])
        gross_losses = np.abs(np.sum(active_trades[active_trades < 0]))
        profit_factor = gross_profits / gross_losses if gross_losses > 0 else np.inf

        # Calculate daily compounding returns assuming a 5% allocation per trade
        allocation_pct = 0.05
        margin_required = 5.0 # Spread Width
        num_contracts = (self.initial_capital * allocation_pct) / (margin_required * 100)

        daily_dollar_pnl = active_trades * num_contracts * 100
        equity_curve = self.initial_capital + np.cumsum(daily_dollar_pnl)

        # Drawdown profile
        running_max = np.maximum.accumulate(equity_curve)
        drawdowns = (equity_curve - running_max) / running_max
        max_dd = np.min(drawdowns)

        # Risk-adjusted performance metrics
        returns_pct = daily_dollar_pnl / self.initial_capital
        sharpe = np.sqrt(252) * np.mean(returns_pct) / np.std(returns_pct) if np.std(returns_pct) > 0 else 0

        downside_std = np.std(returns_pct[returns_pct < 0])
        sortino = np.sqrt(252) * np.mean(returns_pct) / downside_std if downside_std > 0 else np.inf

        metrics = {
            "Total Trades": len(active_trades),
            "Win Rate": f"{win_rate * 100:.2f}%",
            "Profit Factor": f"{profit_factor:.2f}",
            "Max Drawdown": f"{max_dd * 100:.2f}%",
            "Sharpe Ratio": f"{sharpe:.2f}",
            "Sortino Ratio": f"{sortino:.2f}",
            "Final Equity": f"${equity_curve[-1]:,.2f}"
        }
        return metrics, df

# Execution block
if __name__ == "__main__":
    engine = LowDTEVectorEngine()
    # Build 4 years of simulated high-frequency price paths
    data = engine.generate_synthetic_data(num_days=1000)

    # Run a systematic 0DTE Short Put Spread backtest
    metrics, results_df = engine.run_short_credit_spread_backtest(
        data,
        target_delta=0.15,   # Out-of-the-money safety cushion
        spread_width=5.0,    # Hard defined risk cap
        stop_loss_mult=3.0,  # Exit if premium hits 3x credit
        vix_filter=22.0      # Stay out of high-volatility environments
    )

    print("=== PERFORMANCE METRICS ===")
    for key, value in metrics.items():
        print(f"{key:15}: {value}")

=== PERFORMANCE METRICS ===
Total Trades   : 979
Win Rate       : 91.01%
Profit Factor  : 5.13
Max Drawdown   : -1.84%
Sharpe Ratio   : 13.40
Sortino Ratio  : 42.47
Final Equity   : $4,324,926.31
